# 04 – DQ Export (one CSV per rule, for SharePoint)
For every active rule, writes the **full output of its failure query** (all columns, no DISTINCT, no record_key join)
to one CSV file:

- `Files/dq_exports/latest/DQ_Rule_<id>_<rule_name>.csv` – folder is cleared and rewritten each run;
  the pipeline copies the whole folder to SharePoint with one Copy activity
- `Files/dq_exports/archive/<date>/` – dated copy of the same files, kept `EXPORT_ARCHIVE_DAYS` days

Run it in the pipeline **after** `03_dq_runner`. It does not change any DQ table.

Note: file row counts are query **rows**; dashboard counts are distinct **records** (record_key). They can differ.

In [ ]:
%run 00_dq_config

In [ ]:
# Step 1: prepare folders (clear 'latest' so files of deactivated/renamed rules disappear)
import os, re, shutil
from datetime import timedelta

export_date = now_local().date()
archive_dir = f"{EXPORT_ARCHIVE_DIR}/{export_date}"

shutil.rmtree(EXPORT_LATEST_DIR, ignore_errors=True)
os.makedirs(EXPORT_LATEST_DIR, exist_ok=True)
shutil.rmtree(archive_dir, ignore_errors=True)       # rerun on the same day replaces that day's archive
os.makedirs(archive_dir, exist_ok=True)

In [ ]:
# Step 2: export each active rule's failing rows
rules = (spark.table(T_CATALOG)
              .filter("is_active = true")
              .orderBy("rule_id")
              .collect())

exported, errored = 0, 0

for r in rules:
    try:
        if r.failure_condition:
            fail = spark.sql(r.base_sql).filter(r.failure_condition)
        else:
            fail = spark.sql(r.failure_sql)

        safe_name = re.sub(r"[^A-Za-z0-9]+", "_", r.rule_name).strip("_")[:80]
        file_name = f"{EXPORT_FILE_PREFIX}_{r.rule_id:03d}_{safe_name}.csv"
        path      = f"{EXPORT_LATEST_DIR}/{file_name}"

        pdf = fail.toPandas()     # all columns of the rule's query, no DISTINCT
        # utf-8-sig so Excel shows non-English characters; empty result still writes a header-only file
        pdf.to_csv(path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
        shutil.copyfile(path, f"{archive_dir}/{file_name}")

        exported += 1
        print(f"Rule {r.rule_id}: {len(pdf):>8} rows -> {file_name}")
    except Exception as e:
        errored += 1
        print(f"Rule {r.rule_id} export FAILED: {str(e)[:300]}")

print(f"Exported {exported} files, {errored} errors")

In [ ]:
# Step 3: delete dated archive folders older than EXPORT_ARCHIVE_DAYS
cutoff = export_date - timedelta(days=EXPORT_ARCHIVE_DAYS)

for d in os.listdir(EXPORT_ARCHIVE_DIR):
    try:
        if datetime.strptime(d, "%Y-%m-%d").date() < cutoff:
            shutil.rmtree(f"{EXPORT_ARCHIVE_DIR}/{d}", ignore_errors=True)
            print(f"Deleted old archive folder: {d}")
    except ValueError:
        pass   # not a dated folder

In [ ]:
# Step 4: return result to the pipeline
notebookutils.notebook.exit(f"exported={exported}; errors={errored}")